In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.metrics import average_precision_score
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras import layers, Model

In [ ]:
CSV_PATH = "/content/drive/MyDrive/iot/train_test_network (2).csv"  
df = pd.read_csv(CSV_PATH)

assert "name" in df.columns
assert "Group" in df.columns
df["name"] = df["name"].astype(str)
df["Group"] = pd.to_numeric(df["Group"], errors="coerce").astype("Int64")

In [3]:
time_candidates = ["ts","time","start_time","timestamp","frame.time_epoch"]
time_col = next((c for c in time_candidates if c in df.columns), None)
if time_col is None:
    df["_row_order"] = np.arange(len(df))
    time_col = "_row_order"
df[time_col] = pd.to_numeric(df[time_col], errors="coerce").fillna(0.0)
df = df.sort_values(["Group", time_col]).reset_index(drop=True)


In [4]:
num_candidates = [
    "duration","src_bytes","dst_bytes","src_pkts","dst_pkts",
    "src_ip_bytes","dst_ip_bytes","missed_bytes",
    "dns_qtype","dns_rcode","dns_qclass"
]
num_cols = [c for c in num_candidates if c in df.columns]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce").replace([np.inf,-np.inf], np.nan).fillna(0.0)

# derived numeric (very helpful)
eps = 1e-6
df["tot_bytes"] = df.get("src_bytes",0) + df.get("dst_bytes",0)
df["tot_pkts"]  = df.get("src_pkts",0)  + df.get("dst_pkts",0)
df["bytes_ratio"] = df.get("src_bytes",0) / (df.get("dst_bytes",0) + 1.0)
df["pkts_ratio"]  = df.get("src_pkts",0)  / (df.get("dst_pkts",0) + 1.0)
df["byte_rate"]   = df["tot_bytes"] / (df.get("duration",0) + 1e-3)
df["pkt_rate"]    = df["tot_pkts"]  / (df.get("duration",0) + 1e-3)
df["avg_bytes_per_pkt"] = df["tot_bytes"] / (df["tot_pkts"] + 1.0)

for c in ["duration","tot_bytes","tot_pkts","byte_rate","pkt_rate"]:
    df[f"log1p_{c}"] = np.log1p(df[c].clip(lower=0))

extra_num = ["tot_bytes","tot_pkts","bytes_ratio","pkts_ratio","byte_rate","pkt_rate",
             "avg_bytes_per_pkt","log1p_duration","log1p_tot_bytes","log1p_tot_pkts","log1p_byte_rate","log1p_pkt_rate"]
num_cols2 = num_cols + [c for c in extra_num if c in df.columns]

In [5]:
cat_candidates = [
    "proto","service","conn_state",
    "dns_AA","dns_RD","dns_RA","dns_rejected",
    "ssl_version","ssl_cipher","ssl_resumed","ssl_established"
]
cat_cols = [c for c in cat_candidates if c in df.columns]

# ports -> bucket (比直接数字更有效)
for pcol in ["dst_port","src_port"]:
    if pcol in df.columns:
        df[pcol] = pd.to_numeric(df[pcol], errors="coerce").fillna(-1).astype(int)
    else:
        df[pcol] = -1

def port_bucket(p):
    if p == 53: return "p53_dns"
    if p == 80: return "p80_http"
    if p == 443: return "p443_https"
    if p == 123: return "p123_ntp"
    if p == 1883: return "p1883_mqtt"
    if p == 5683: return "p5683_coap"
    if p == -1: return "pNA"
    if p < 1024: return "pWellKnownOther"
    return "pHigh"

df["dst_port_bucket"] = df["dst_port"].map(port_bucket)
df["src_port_bucket"] = df["src_port"].map(port_bucket)
cat_cols += ["dst_port_bucket","src_port_bucket"]

# optional: coarse IP bucket (容易泄漏环境信息；如果你只想拉高分可以加，但不“NDSS严格”)
# df["src_ip"] = df["src_ip"].astype(str).fillna("-") if "src_ip" in df.columns else "-"
# df["dst_ip"] = df["dst_ip"].astype(str).fillna("-") if "dst_ip" in df.columns else "-"
# cat_cols += [c for c in ["src_ip","dst_ip"] if c in df.columns]

for c in cat_cols:
    df[c] = df[c].astype(str).fillna("-")

print("Numeric dims:", len(num_cols2), num_cols2[:10], "...")
print("Categorical dims:", len(cat_cols), cat_cols)

# ---------- 3) scale + one-hot ----------
scaler = StandardScaler()
Xn = scaler.fit_transform(df[num_cols2].values)

ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
Xc = ohe.fit_transform(df[cat_cols])

X_feat = np.hstack([Xn, Xc]).astype(np.float32)
print("Final per-connection feature_dim =", X_feat.shape[1])

Numeric dims: 23 ['duration', 'src_bytes', 'dst_bytes', 'src_pkts', 'dst_pkts', 'src_ip_bytes', 'dst_ip_bytes', 'missed_bytes', 'dns_qtype', 'dns_rcode'] ...
Categorical dims: 13 ['proto', 'service', 'conn_state', 'dns_AA', 'dns_RD', 'dns_RA', 'dns_rejected', 'ssl_version', 'ssl_cipher', 'ssl_resumed', 'ssl_established', 'dst_port_bucket', 'src_port_bucket']
Final per-connection feature_dim = 89


In [11]:
le = LabelEncoder()
y_type = le.fit_transform(df["name"].values)
g = df["Group"].astype(int).values

SEQ_LEN = 25
STRIDE = 25
MIN_CONN_PER_GROUP = 20

X_seq, y_seq, g_seq = [], [], []
for grp, idxs in df.groupby("Group").indices.items():
    idxs = np.array(idxs)
    if len(idxs) < MIN_CONN_PER_GROUP:
        continue
    y_g = int(pd.Series(y_type[idxs]).mode().iloc[0])

    for s in range(0, len(idxs), STRIDE):
        win = idxs[s:s+SEQ_LEN]
        if len(win) == 0:
            continue
        if len(win) < SEQ_LEN:
            pad = np.zeros((SEQ_LEN-len(win), X_feat.shape[1]), dtype=np.float32)
            seq = np.vstack([X_feat[win], pad])
        else:
            seq = X_feat[win]
        X_seq.append(seq)
        y_seq.append(y_g)
        g_seq.append(int(grp))

X_seq = np.stack(X_seq, axis=0)
y_seq = np.array(y_seq)
g_seq = np.array(g_seq)

print("Sequence dataset:", X_seq.shape, "classes:", len(le.classes_), "groups:", len(np.unique(g_seq)))


Sequence dataset: (8449, 25, 89) classes: 8 groups: 13


In [12]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
tr, te = next(gss.split(X_seq, y_seq, groups=g_seq))

Xtr, Xte = X_seq[tr], X_seq[te]
ytr, yte = y_seq[tr], y_seq[te]

# class weights (macro-AUCPR 友好)
classes = np.unique(ytr)
cw = compute_class_weight(class_weight="balanced", classes=classes, y=ytr)
class_weight = {int(c): float(w) for c, w in zip(classes, cw)}

In [13]:
n_feat = X_seq.shape[2]
n_cls = len(le.classes_)

inp = layers.Input(shape=(SEQ_LEN, n_feat))
x = layers.Masking(mask_value=0.0)(inp)
x = layers.Bidirectional(layers.LSTM(128, return_sequences=True))(x)
x = layers.GlobalMaxPooling1D()(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.35)(x)
out = layers.Dense(n_cls, activation="softmax")(x)

model = Model(inp, out)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy")

cb = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5),
]

history = model.fit(
    Xtr, ytr,
    validation_split=0.1,
    epochs=25,
    batch_size=64,
    callbacks=cb,
    class_weight=class_weight,
    verbose=2
)

proba = model.predict(Xte, batch_size=256, verbose=0)

Epoch 1/25


/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:965: UserWarning: Layer 'global_max_pooling1d_2' (of type GlobalMaxPooling1D) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


109/109 - 19s - 175ms/step - loss: 0.4898 - val_loss: 5.3687 - learning_rate: 1.0000e-03
Epoch 2/25
109/109 - 14s - 125ms/step - loss: 0.3042 - val_loss: 5.4116 - learning_rate: 1.0000e-03
Epoch 3/25
109/109 - 13s - 123ms/step - loss: 0.2944 - val_loss: 6.8335 - learning_rate: 1.0000e-03
Epoch 4/25
109/109 - 12s - 114ms/step - loss: 0.2693 - val_loss: 7.7277 - learning_rate: 5.0000e-04


In [17]:
classes_in_test = np.unique(yte)
proba_sub = proba[:, classes_in_test]
Yte_sub = np.eye(n_cls)[yte][:, classes_in_test]
auprc = average_precision_score(Yte_sub, proba_sub, average="macro")

print("AUCPR =", auprc)

AUCPR = 0.7137137878881329
